In [17]:
from pathlib import Path
import os

with (Path(os.environ['HOME']) / 'wandb.txt').open() as fp:
    key = fp.read().strip()

os.environ['WANDB_API_KEY'] = key
os.environ['WANDB_PROJECT'] = 'qwen_dp'
os.environ['WANDB_LOG_MODEL'] = 'false'
os.environ['WANDB_NAME'] = 'test_dp'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import datasets
import evaluate
import pandas as pd
import numpy as np
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer, BitsAndBytesConfig, DataCollatorWithPadding
from transformers.integrations import WandbCallback
from trl import SFTTrainer, SFTConfig
import torch
from tqdm import tqdm
import wandb

import sys
sys.path.append('../../../../')
from common import new_seed
from task.prop import PropTask, full_text_ds_path, or_text_ds_path

model_name = "Qwen/Qwen2.5-Coder-7B"

wandb.login()


True

In [13]:
def make_ds(depth, split):
    task = PropTask(depth=depth, split=split, cot='text', ds_path=or_text_ds_path)
    task.load_ds()

    ds = datasets.concatenate_datasets([task.true_ds, task.false_ds]).shuffle()

    # temporarily reduce size for debugging
    ds = ds.select(range(5000))

    # TODO: reformat permanently in dataset
    # ds = ds.map(lambda x: {'text': x['prompt'] + x['completion']}, num_proc=16)
    ds = ds.rename_column('prompt', 'text')
    # ds = ds.rename_column('is_true', 'label')
    ds = ds.map(lambda x: {'label': 1 if x['is_true'] else 0}, num_proc=16) 
    ds = ds.remove_columns(['completion'])
    return ds


train_split = 6
# test_splits = [2, 4, 6, 10]
test_splits = [4, 6, 8]
range_hops = [1] + [h + 1 for h in test_splits] + [np.inf]
ranges = list(zip(range_hops[:-1], range_hops[1:]))

train_ds = make_ds(train_split, 'train')
test_ds = make_ds(train_split, 'test')
test_ds = test_ds.select(range(100))  # TODO: should preferably incorporate logic by subsampling during training

val_ds_set = [make_ds(r, split='range') for r in ranges]

Map (num_proc=16):   0%|          | 0/5000 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/5000 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/5000 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/5000 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/5000 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/5000 [00:00<?, ? examples/s]

In [14]:
train_ds[0]

{'is_true': True,
 'length': 5,
 'ops': ['intro h', 'exact', 'apply Or'],
 'text': '<state id="0"><if>p1 p2 p3 : Prop</if><then>⊢ p2 → ((p2 ∨ p3) ∨ p3 ∨ p2 ∨ p2 ∨ p2) ∨ p1</then></state>|',
 'label': 1}

In [15]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

def to_toks(examples):
    return tokenizer(examples['text'])

train_ds = train_ds.map(to_toks, batched=True, num_proc=16)
test_ds = test_ds.map(to_toks, batched=True, num_proc=16)

Map (num_proc=16):   0%|          | 0/5000 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/100 [00:00<?, ? examples/s]

In [18]:
accuracy = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [19]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4'
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    dtype=torch.bfloat16,
    device_map='auto',
    attn_implementation='flash_attention_2',
    quantization_config=quant_config
)

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=("q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj")
)

model = get_peft_model(model, peft_config)

args = TrainingArguments(
    output_dir="~/scratch/qwen25_coder7b_prop_qlora_dp",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    gradient_accumulation_steps=1,      
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=5,
    save_steps=1000,
    bf16=True,                           
    gradient_checkpointing=True,
    optim="adamw_bnb_8bit",
    # optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    weight_decay=0.0,
    # completion_only_loss=True,
    # packing=True,
    # max_length=2048,
    eval_strategy='steps',
    torch_compile=False,
    # report_to=None   
)

trainer = Trainer(
    model=model,
    args=args,
    processing_class=tokenizer,
    data_collator=collator,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    # eval_dataset=train_ds.select(range(100)),
    compute_metrics=compute_metrics
)


def evaluate(full=False):
    model.eval()
    with torch.no_grad():
        all_res = {}
        for r, val_ds in tqdm(zip(ranges, val_ds_set), total=len(val_ds_set)):
            ds = val_ds
            if not full:
                ds = ds.shuffle().select(range(100))

            inp_ids = [tokenizer(text) for text in ds['text']]
            labels = torch.tensor(ds['label'], device='cuda')
            inp = collator(inp_ids)

            inp = {k: v.to('cuda') for k, v in inp.items()}
            out = model(**inp)

            preds = out.logits.argmax(-1)
            acc = torch.mean((preds == labels).float())
            all_res[f'range_{r}'] = {'gen_acc': acc}

            
    model.train()
    return all_res


# TODO: generalize to evaluate on multiple dataset splits
class WandbEvalCallback(WandbCallback):
    def __init__(self, trainer, val_ds_set, num_samples=100):
        super().__init__()
        self.trainer = trainer
        self.tokenizer = self.trainer.processing_class
        self.val_ds_set = val_ds_set
        self.num_samples = num_samples
        
        self.succ_id = self.tokenizer.encode('success')[0]
        self.fail_id = self.tokenizer.encode('failure')[0]
        

    def on_evaluate(self, args, state, control, **kwargs):
        super().on_evaluate(args, state, control, **kwargs)
        all_res = evaluate()
        self._wandb.log(all_res)

        
eval_callback = WandbEvalCallback(trainer, val_ds_set)
trainer.add_callback(eval_callback)

trainer.train()
wandb.finish()

final_res = evaluate(full=False) # TODO: may require batching for full evaluation
df = pd.DataFrame([{
    'name': 'DP Qwen',
    'train_hop': train_split,
    'info': final_res
}])

df.to_pickle(f'res.{new_seed()}.pkl')

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen2.5-Coder-7B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


/n/netscratch/pehlevan_lab/Lab/wlt/envs/venv_imply/lib/python3.12/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss,Validation Loss,Accuracy
5,0.000100,0.699385,0.940000
10,0.000000,1.046250,0.940000
15,0.000000,1.272500,0.940000
20,0.000000,1.368750,0.940000
25,0.000000,1.402500,0.940000
30,0.000000,1.417500,0.940000
35,0.000000,1.420000,0.940000
40,0.000000,1.423750,0.940000
45,0.000000,1.427500,0.940000
50,0.000000,1.428750,0.940000


100%|██████████| 4/4 [00:01<00:00,  2.92it/s]
/n/netscratch/pehlevan_lab/Lab/wlt/envs/venv_imply/lib/python3.12/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
100%|██████████| 4/4 [00:01<00:00,  2.99it/s]
/n/netscratch/pehlevan_lab/Lab/wlt/envs/venv_imply/lib/python3.12/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
100%|██████████| 4/4 [00:01<00:00,  2.97it/s]
/n/netscratch/pehlevan_lab/Lab/wlt/envs/venv_imply/lib/python3.12/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
100%|██████████| 4/4 [00:01<00:00,  2.97it/s]
/n/netscratch/pehlevan_lab/Lab/wlt/envs/venv_imply/lib/python3.12/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  war

eval/accuracy,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/loss,▁▁▄▄▇▇██████████████████████████████████
eval/runtime,█▁▁▂▂▂▂▁▁▁▂▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▃▂▂▂▂▂
eval/samples_per_second,▁▁██▇▇▇█████▇▇████▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▆▇▇▇▇▇
eval/steps_per_second,▁▁██▇▇▇████▇▇██████▇▇▇▇▇▇▇▇▇▇▇▇▇▇▆▇▇▇▇▇▇
train/epoch,▁▁▁▁▁▂▂▂▂▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇███
train/global_step,▁▁▁▁▂▂▂▂▂▂▂▂▂▃▃▃▄▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇██
train/grad_norm,██▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/learning_rate,▄████████▇▇▇▇▇▇▆▆▅▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁
train/loss,██▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/accuracy,0.94


100%|██████████| 4/4 [00:01<00:00,  2.98it/s]


In [16]:
df

,name,train_hop,info
0,DP Qwen,6,"{'range_(1, 3)': {'gen_acc': tensor(0.6700, de..."


In [ ]:
model.base_model.model.score  # TODO: identify which module to modify <-- STOPPED HERE

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4'
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    dtype=torch.bfloat16,
    device_map='auto',
    attn_implementation='flash_attention_2',
    quantization_config=quant_config
)

model

In [ ]:
model = get_peft_model(model, peft_config)
model

In [ ]:
model.score.original_module.weight

In [ ]:
ds = test_ds.select(range(10))
ids = [trainer.processing_class(text) for text in ds['text']]
tok_ds = datasets.Dataset.from_list(ids)

trainer.args.packing = True
pred = trainer.predict(tok_ds)

In [ ]:
from transformers import DataCollator

trainer.processing_class.padding_side = 'left'
inp = DataCollatorWithPadding(tokenizer=trainer.processing_class)(ids)
inp

In [ ]:
inp_ids_orig = inp['input_ids']
inp['input_ids'] = inp['input_ids'][:,:-500]
inp['input_ids']

In [ ]:
inp['attention_mask'].dtype

In [ ]:
with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
    inp['input_ids'] = inp['input_ids'].to(device='cuda')
    inp['attention_mask'] = inp['attention_mask'].to(device='cuda')
    out = trainer.model.generate(**inp, max_new_tokens=2048)

In [ ]:
trainer.processing_class.encode('<success />')

In [ ]:
succ_id = trainer.processing_class.encode('success')[0]
fail_id = trainer.processing_class.encode('failure')[0]

labels = DataCollatorWithPadding(tokenizer=trainer.processing_class)(ids)
labels = labels['input_ids'].to('cuda')

preds = out

def score(preds, labels, succ_id, fail_id):
    is_true = torch.argmax((labels == succ_id).int(), axis=-1) > 0

    t = torch.argmax((preds == succ_id).int(), axis=-1)
    f = torch.argmax((preds == fail_id).int(), axis=-1)

    pred_is_true = (t != 0) * ((f == 0) + (t < f))
    pred_is_false = (f != 0) * ((t == 0) + (f < t))

    true_pos = is_true * pred_is_true
    true_neg = (~is_true) * pred_is_false
    false_pos = (~is_true) * pred_is_true
    false_neg = is_true * pred_is_false

    true_pos = torch.mean(true_pos.float())
    true_neg = torch.mean(true_neg.float())
    false_pos = torch.mean(false_pos.float())
    false_neg = torch.mean(false_neg.float())
    
    return {
        'true_pos': true_pos,
        'true_neg': true_neg,
        'false_pos': false_pos,
        'false_neg': false_neg
    }


score(preds, labels, succ_id, fail_id)

In [ ]:
trainer.processing_class.batch_decode(out)[0]

In [ ]:
trainer.processing_class.decode(inp['input_ids'][0][:-700])